### Create Silver company info

We calculate market‑cap deciles to segment companies into comparable size groups.
This transformation converts raw market‑cap values into 10 percentile buckets.
It allows Power BI users to filter the dataset by company size — for example, showing only the top 10% largest companies.

Companies with missing market‑cap values are removed to ensure accurate percentile calculations.

In [3]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

bronze_clean = (
    spark.table("bronze_company_info")
    .filter(F.col("market_cap").isNotNull())
    .dropDuplicates(["cik"])
)


# replace NaN with null
numeric_types = ["double", "float", "integer", "bigint", "long", "decimal"]

cleaned_cols = []
for c, dtype in bronze_clean.dtypes:
    if dtype in numeric_types:
        cleaned_cols.append(
            F.when(
                F.isnan(F.col(c)) |
                (F.col(c) == float("inf")) |
                (F.col(c) == float("-inf")),
                None
            ).otherwise(F.col(c)).alias(c)
        )
    else:
        cleaned_cols.append(F.col(c))

df = bronze_clean.select(cleaned_cols)

# Ranking: 1 = highest market cap
w_rank = Window.orderBy(F.col("market_cap").desc())

# Decile: 0–9 (ntile gives 1–10)
w_ntile = Window.orderBy(F.col("market_cap").asc())

silver = (
    bronze_clean
    # ntile gives 1–10 → wir substract 1 → 0–9
    .withColumn("market_cap_decile", F.ntile(10).over(w_ntile) - 1)

    # Bucket rom Decile
    .withColumn(
        "market_cap_bucket",
        F.concat(
            (F.col("market_cap_decile") * 10).cast("string"),
            F.lit("% – "),
            ((F.col("market_cap_decile") + 1) * 10).cast("string"),
            F.lit("%")
        )
    )

    # Rank: 1 = highest Market Cap
    .withColumn("market_cap_rank", F.row_number().over(w_rank))
)

silver.write.format("delta").mode("overwrite").saveAsTable("silver_company_info")


StatementMeta(, fa5834cc-d9f1-4819-b141-134ee97c96a7, 5, Finished, Available, Finished, False)